This notebook is for extracting scenario events(injects) and discussion questions from our chosen CISA manuals.

**Data source 1: CISA Tabletop Exercise Packages**


We have chosen the files:

**healthcare_and_public_health.docx** | Contains: phishing, malfunctioning medical devices and incorrect patient records, ransomware and a patient-data breach

**vendor_supply_chain_compromise.docx** | Contains: Malware, Company relies on an outside IT provider(MSP). That provider gets hacked and the company's network goes down and later it turns out the providers software spread malware.

**ransomware.docx** | Contains: Phishing, a stolen laptop and unusual data transfers, leading to ransomware, leaked employee data and media attention

**insider_threat.docx** | Contains: Accidental data disclosure and a fired employee, followed by a rogue admin account and deleted files

**executives_and_senior_leadership.docx** | Contains: Two short vignettes for leaders: whaling/ransomware and an insider stealing data

**industrial_control_systems.docx** | Contains: Industrial plant (IT + OT): phishing and MFA fatigue lead to ransomware that shuts down operations

**financial_services.docx** | Contains: Bank: cloud provider data theft, then ransomware that takes down online banking 

**communications.docx** | Contains: Rural broadband provider: a DDoS attack causes internet outages and overloads the call centre

#### 1. Getting the Files
   
First we need to get the actual files and were using our repositorium on github for that. This was our best option to keep the same files across multiple devices.

In [3]:
%pip install python-docx

import re
import io
import requests
import pandas as pd
from docx import Document
from pathlib import Path

# Defining the repositorium link
REPO = "Mastercoder-1000/Data-Strategy-amp-Modelling-for-a-Strategic-Cybersecurity-Decision-Making-Exercise-Platform-"
# Defining the branch were working in 
BRANCH = "main"
# and the raw url 
RAW_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"

# For each file were reading it gets a shortened name and a prefix
# The prefix is put in front of every ID that is later used in the system.

files = {
    "Healthcare": ("healthcare_and_public_health.docx", "HC"),
    "Vendor": ("vendor_supply_chain_compromise.docx", "VD"),
    "Ransomware": ("ransomware.docx", "RW"),
    "Insider Threat": ("insider_threat.docx", "IT"),
    "Executives": ("executives_and_senior_leadership.docx", "EX"),
    "Industrial Control Systems": ("industrial_control_systems.docx", "ICS"),
    "Financial Services": ("financial_services.docx", "FS"),
    "Communications": ("communications.docx", "CM"),
}

Note: you may need to restart the kernel to use updated packages.


#### 2. Downloading the manuals
Now were downloading each of the manuals from `data/raw/` in the repository. And were keeping it in the memory.

Were deliberately avoiding using local file paths. So that the same code works for each of our teammates. 

In [2]:
# First we will create an empty dictionary, which will later store the manual name and it's file contents
manual_bytes = {}

#Now ere looping through all of the files in files
#(filename, _) - this contains the _ to tell Python that we dont need the prefix right now
# so for each of the files loop through their names and their corresponding filename and read the info in
for name, (filename, _) in files.items():
    #Building the web address line, the RAW_URL was defined in the previous cell
    url = f"{RAW_URL}/data/raw/{filename}"
    # This is the file download part and we added a timeout so the cell doenst run forever if the server isnt responding.
    response = requests.get(url, timeout=60)
    # if the download failed stop the notebook otherwise continue
    response.raise_for_status()
    #Now were using the manual_bites file we created before to store the file we got in from the internet, with its shortened name and into manual_bytes 
    manual_bytes[name] = response.content
    # This is a printing line just  to see if all the files were downloaded
    print(f"{name}: {len(response.content) / 1_000_000:.1f} MB")


def open_manual(name):
    """Opens a downloaded manual as a Word document (from memory, not from disk)."""
    return io.BytesIO(manual_bytes[name])

Healthcare: 4.6 MB
Vendor: 4.5 MB
Ransomware: 4.6 MB
Insider Threat: 5.7 MB
Executives: 4.5 MB
Industrial Control Systems: 7.1 MB
Financial Services: 5.9 MB
Communications: 6.2 MB


#### 3. Extracting the data
A function is used to identify scenario events and discussion questions in the CISA manuals.

*Why:* this function is unchanged from the previous version. `Document()` accepts an in-memory file as well as a file path, so it works with the downloaded manuals without any changes.

In [4]:
def parse_manual(path, manual_name, prefix):
    doc = Document(path)

    # keep only the paragraphs that aren't empty
    paragraphs = [
        (i, p.text.strip())
        for i, p in enumerate(doc.paragraphs)
        if p.text.strip()
    ]

    events = []
    questions = []

    section = None # current module or vignette
    day = None # current scenario day
    mode = None # whether we're reading events or questions
    event_parts = [] # paragraphs belonging to the current event
    event_index = None # paragraph index where the event starts
    started = False # whether we've reached the actual exercise

    def save_event():
        if event_parts and section is not None:
            events.append({
                "manual_title": manual_name,
                "inject_id": f"{prefix}-INJ-{event_index}",
                "section": section,
                "day": day,
                "event_description": " ".join(event_parts),
                "source_paragraph": event_index
            })

    for idx, text in paragraphs:

        # find the beginning of the actual exercise
        if not started:
            if manual_name == "Executives":
                start_heading = "General Discussion Questions"
            else:
                start_heading = "Module 1"

            if text == start_heading:
                started = True
                section = text

                if manual_name == "Executives":
                    mode = "question"
                else:
                    mode = "event"

            continue

        # stop before the appendices
        if re.fullmatch(
            r"Appendix\s+A\s*:.*",
            text,
            re.I
        ):
            save_event()
            break

        # recognize modules and vignettes
        is_module = re.fullmatch(
            r"Module\s+[12](?:\s*[–-].*)?",
            text,
            re.I
        )

        is_vignette = re.fullmatch(
            r"Vignette\s+[12]\s*:.*",
            text,
            re.I
        )

        if is_module or is_vignette:
            save_event()

            section = text
            day = None
            mode = "event"
            event_parts = []
            event_index = None
            continue

        # recognize discussion question headings
        if text == "Discussion Questions":
            save_event()
            event_parts = []
            mode = "question"
            continue

        # recognize scenario days
        if re.fullmatch(r"Day\s+\d+.*", text, re.I):
            save_event()

            day = text
            event_index = idx
            event_parts = []
            mode = "event"
            continue

        # recognize scenario headings in Executives manual
        if text == "Scenario" and manual_name == "Executives":
            mode = "event"
            continue

        # skip introductory descriptions of questions
        if (
            text.lower().startswith(
                "discussion questions included"
            )
            or text.lower().startswith(
                "the general discussion questions below"
            )
        ):
            continue

        # recognize questions and discussion prompts
        is_prompt = (
            "?" in text
            or bool(re.match(
                r"^(Describe|Discuss|Explain|Identify|List|"
                r"Based on|Using your|Utilizing your|In this scenario|What|How|Who|When|"
                r"Where|Does|Do|Are|Is|Have|Has|"
                r"Can|Could|Would|Should)\b",
                text,
                re.I
            ))
        )

        # if a question follows an event, switch modes
        if mode == "event" and is_prompt:
            save_event()
            event_parts = []
            mode = "question"

        # save discussion questions
        if mode == "question":
            if is_prompt:
                questions.append({
                    "manual_title": manual_name,
                    "question_id": f"{prefix}-Q-{idx}",
                    "section": section,
                    "day": day,
                    "question_text": text,
                    "domain_category": None,
                    "difficulty_level": None,
                    "required_skill": None,
                    "source_paragraph": idx
                })

        # save event descriptions
        elif mode == "event":
            if manual_name != "Executives" and day is None:
                continue

            if event_index is None:
                event_index = idx

            event_parts.append(text)

    return pd.DataFrame(events), pd.DataFrame(questions)

#### 4. Processing the manuals
The extraction function is applied to all eight manuals, and the results are combined into two DataFrames.

In [5]:
all_events = []
all_questions = []

for name, (filename, prefix) in files.items():
    events, questions = parse_manual(open_manual(name), name, prefix)

    all_events.append(events)
    all_questions.append(questions)

    print(f"{name}: {len(events)} events, {len(questions)} questions")

df_injects = pd.concat(all_events, ignore_index=True)
df_questions = pd.concat(all_questions, ignore_index=True)

print("\nTotal events:", len(df_injects))
print("Total questions:", len(df_questions))

display(df_injects.head())
display(df_questions.head())

Healthcare: 11 events, 61 questions
Vendor: 8 events, 49 questions
Ransomware: 8 events, 59 questions
Insider Threat: 12 events, 53 questions
Executives: 2 events, 49 questions
Industrial Control Systems: 9 events, 59 questions
Financial Services: 8 events, 55 questions
Communications: 8 events, 45 questions

Total events: 66
Total questions: 430


,manual_title,inject_id,section,day,event_description,source_paragraph
0,Healthcare,HC-INJ-87,Module 1,Day 1,The Cybersecurity and Infrastructure Security ...,87
1,Healthcare,HC-INJ-89,Module 1,Day 9,<Organization> employees receive e-mail notifi...,89
2,Healthcare,HC-INJ-117,Module 1,Day 10,Manufacturer’s updates are pushed for installa...,117
3,Healthcare,HC-INJ-119,Module 1,Day 13,A third-party electronic medical record (EMR) ...,119
4,Healthcare,HC-INJ-126,Module 1,Day 15,Radiology reports an issue with their Magnetic...,126


,manual_title,question_id,section,day,question_text,domain_category,difficulty_level,required_skill,source_paragraph
0,Healthcare,HC-Q-93,Module 1,Day 9,What are the greatest cyber threats to your or...,None,None,None,93
1,Healthcare,HC-Q-94,Module 1,Day 9,What cybersecurity threat information does you...,None,None,None,94
2,Healthcare,HC-Q-95,Module 1,Day 9,What cyber threat information is most useful?,None,None,None,95
3,Healthcare,HC-Q-96,Module 1,Day 9,How is information disseminated throughout you...,None,None,None,96
4,Healthcare,HC-Q-97,Module 1,Day 9,What actions would your organization take in r...,None,None,None,97


#### 5. Checking the extracted data
The extracted questions **and events** are compared with the original documents, and record IDs are checked for duplicates.

*Why:* previously only the questions were checked. An event's `source_paragraph` points to its "Day" heading, so the check confirms that the event text appears in the document after that heading.

In [6]:
documents = {name: Document(open_manual(name)) for name in files}

# questions must match their source paragraph exactly
for _, row in df_questions.iterrows():
    doc = documents[row["manual_title"]]
    original = doc.paragraphs[int(row["source_paragraph"])].text.strip()
    assert original == row["question_text"], f"Mismatch: {row['question_id']}"

# event text must appear in the document after its source paragraph
for _, row in df_injects.iterrows():
    doc = documents[row["manual_title"]]
    following = " ".join(
        p.text.strip() for p in doc.paragraphs[int(row["source_paragraph"]):] if p.text.strip()
    )
    assert row["event_description"] in following, f"Mismatch: {row['inject_id']}"

assert df_injects["inject_id"].is_unique
assert df_questions["question_id"].is_unique

print("All validation checks passed!")

All validation checks passed!


#### 6. Saving the datasets locally
The extracted events and discussion questions are saved to `data/processed/` in the project folder.

*Why:* the other notebooks (`cisa_analysis`, `exercise_dataset`, ...) read these CSV files, so a local copy is still saved for them. The path is relative to the project folder, so it works on every computer that has the project cloned.

In [7]:
output_dir = Path("../data/processed")
if not Path("../data").exists():
    output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

outputs = {
    "cisa_scenario_injects.csv": df_injects,
    "cisa_discussion_questions.csv": df_questions,
}

for filename, df in outputs.items():
    df.to_csv(output_dir / filename, index=False, encoding="utf-8-sig")
    print("Saved:", (output_dir / filename).resolve())

Saved: C:\Users\noral\Documents\IntroToDataScience\Project\ProjectRepo\data\processed\cisa_scenario_injects.csv
Saved: C:\Users\noral\Documents\IntroToDataScience\Project\ProjectRepo\data\processed\cisa_discussion_questions.csv
